# Math Foundations for Machine Learning

This notebook walks through the core math concepts behind ML, using the
`agentexplorr.foundations` module. Every concept is implemented from scratch
so you can see the numbers and build intuition.

**Topics covered:**
1. Activation functions and their derivatives
2. Loss functions with gradient derivations
3. Optimizers (SGD, Momentum, Adam)
4. Backpropagation through a computational graph
5. Linear algebra essentials (dot products, norms, SVD)

**Prerequisites:** Basic Python. We explain all the math.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import everything from our foundations module
from agentexplorr.foundations import (
    relu, sigmoid, tanh, gelu, leaky_relu, softmax,
    mean_squared_error, cross_entropy_loss, binary_cross_entropy, huber_loss,
    SGD, Momentum, Adam, AdamW,
    ComputationalGraph, Node,
    LinearAlgebraTeacher,
)

print('All imports successful!')

## 1. Activation Functions

Activation functions introduce **non-linearity** into neural networks.
Without them, stacking linear layers would just produce another linear
function — no matter how deep the network.

In [ ]:
x = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Activation Functions', fontsize=16)

activations = [
    ('ReLU', relu),
    ('Sigmoid', sigmoid),
    ('Tanh', tanh),
    ('GELU', gelu),
    ('Leaky ReLU', leaky_relu),
    ('Softmax (1D demo)', lambda x: sigmoid(x)),  # softmax on 1D ~ sigmoid
]

for ax, (name, fn) in zip(axes.flat, activations):
    y = fn(x)
    ax.plot(x, y, 'b-', linewidth=2)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Key takeaways:
- **ReLU** is the default choice for hidden layers (fast, no vanishing gradient for x > 0)
- **GELU** is used in Transformers (GPT, BERT) — smoother than ReLU
- **Sigmoid** squashes to (0, 1) — good for binary probabilities
- **Softmax** normalizes to a probability distribution — used in output layers for classification

## 2. Loss Functions

Loss functions measure how "wrong" the model is. The optimizer's job is
to minimize this number.

In [ ]:
y_true = np.array([1.0, 0.0, 1.0, 0.0, 1.0])
y_pred = np.array([0.9, 0.1, 0.8, 0.3, 0.95])

print('Predictions vs truth:')
print(f'  True: {y_true}')
print(f'  Pred: {y_pred}')
print()
print(f'  MSE loss:             {mean_squared_error(y_true, y_pred):.6f}')
print(f'  Binary Cross-Entropy: {binary_cross_entropy(y_true, y_pred):.6f}')
print(f'  Huber loss (delta=1): {huber_loss(y_true, y_pred):.6f}')
print()
print('Notice: BCE is much more sensitive to confident wrong predictions.')
print('If we make one bad prediction...')

y_pred_bad = np.array([0.9, 0.1, 0.8, 0.99, 0.95])  # 4th prediction is very wrong!
print(f'\n  Bad pred BCE: {binary_cross_entropy(y_true, y_pred_bad):.6f}  (much higher!)')
print(f'  Bad pred MSE: {mean_squared_error(y_true, y_pred_bad):.6f}  (only slightly higher)')

## 3. Optimizers

Optimizers update model weights to minimize the loss. Let's visualize
how different optimizers descend a 2D loss landscape.

In [ ]:
# Simulate optimizing f(w) = w1^2 + 10*w2^2 (an elongated bowl)
# This is hard for vanilla SGD because the gradients oscillate.

def loss_fn(w):
    return w[0]**2 + 10 * w[1]**2

def grad_fn(w):
    return np.array([2 * w[0], 20 * w[1]])

# Run each optimizer for 50 steps
optimizers = {
    'SGD (lr=0.05)': SGD(lr=0.05),
    'Momentum (0.9)': Momentum(lr=0.05, beta=0.9),
    'Adam (lr=0.3)': Adam(lr=0.3),
}

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot loss landscape contours
w1 = np.linspace(-5, 5, 100)
w2 = np.linspace(-5, 5, 100)
W1, W2 = np.meshgrid(w1, w2)
Z = W1**2 + 10 * W2**2
ax.contour(W1, W2, Z, levels=30, cmap='viridis', alpha=0.5)

for name, opt in optimizers.items():
    w = np.array([4.0, 4.0])
    trajectory = [w.copy()]
    for _ in range(50):
        g = grad_fn(w)
        w = opt.step(w, g)
        trajectory.append(w.copy())
    trajectory = np.array(trajectory)
    ax.plot(trajectory[:, 0], trajectory[:, 1], 'o-', markersize=3, label=name)

ax.set_xlabel('w1')
ax.set_ylabel('w2')
ax.set_title('Optimizer Trajectories on f(w) = w1² + 10·w2²')
ax.legend()
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
plt.show()

### Key takeaways:
- **SGD** oscillates on elongated landscapes
- **Momentum** smooths the oscillations with a velocity term
- **Adam** adapts per-parameter learning rates — converges fastest on this surface
- **AdamW** = Adam + weight decay (the standard for training Transformers)

## 4. Backpropagation

Backpropagation is just the **chain rule** applied systematically to a
computational graph. Let's trace it through a simple example.

In [ ]:
# Build a computational graph: f(x, y) = (x + y) * y
# df/dx = y, df/dy = x + 2y

graph = ComputationalGraph()

x_node = graph.create_input('x', 3.0)
y_node = graph.create_input('y', 2.0)

# sum_node = x + y = 5.0
sum_node = graph.add(x_node, y_node, name='sum')

# product_node = sum * y = 5.0 * 2.0 = 10.0
product_node = graph.multiply(sum_node, y_node, name='product')

# Forward pass
result = graph.forward()
print(f'Forward pass: f(3, 2) = (3 + 2) * 2 = {result}')

# Backward pass (compute gradients)
gradients = graph.backward()
print(f'\nGradients (backpropagation):')
print(f'  df/dx = y = {gradients.get("x", "N/A")}  (expected: 2.0)')
print(f'  df/dy = x + 2y = {gradients.get("y", "N/A")}  (expected: 7.0)')
print(f'\nThis is how PyTorch\'s .backward() works under the hood!')

## 5. Linear Algebra Essentials

Matrix operations are the building blocks of neural networks.
Every forward pass is a series of matrix multiplications.

In [ ]:
teacher = LinearAlgebraTeacher()

# Dot product
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print('--- Dot Product ---')
print(f'  a = {a}')
print(f'  b = {b}')
print(f'  a · b = {teacher.dot_product(a, b)}  (= 1*4 + 2*5 + 3*6 = 32)')

# Matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print('\n--- Matrix Multiplication ---')
print(f'  A @ B = \n{teacher.matrix_multiply(A, B)}')

# Norms
v = np.array([3.0, 4.0])
print(f'\n--- Vector Norms ---')
print(f'  v = {v}')
print(f'  L2 norm (Euclidean): {teacher.l2_norm(v)}  (= sqrt(9 + 16) = 5.0)')

print('\n--- SVD (foundation for LoRA!) ---')
print('SVD decomposes a matrix into U @ S @ Vt')
print('LoRA uses low-rank approximation via truncated SVD')
print('to reduce parameters from d×d to d×r + r×d (where r << d)')

## Next Steps

Now that you understand the math foundations, explore:

1. **`notebooks/04_classical_ml/02_pytorch_basics.ipynb`** — Apply these concepts in PyTorch
2. **`src/agentexplorr/classical_ml/deep_learning/transformer.py`** — See attention use matrix math
3. **`notebooks/03_llm_training/02_lora_fine_tuning.ipynb`** — See SVD (low-rank) in action

Run `python -m agentexplorr foundations` for a quick CLI demo of all foundations.